In [1]:
# IMPORTAÇÃO E CONFIGURAÇÃO

import subprocess
import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt
import ipywidgets as widgets

from pathlib import Path
from IPython.display import Audio, display, clear_output

In [5]:
from pathlib import Path
import sys

ffmpeg_path = (
    Path(sys.prefix)
    / "Library"
    / "bin"
    / "ffmpeg.exe"
)

print("Caminho:", ffmpeg_path)
print("FFmpeg encontrado:", ffmpeg_path.exists())

Caminho: C:\Users\beatr\anaconda\envs\orange_anaconda\Library\bin\ffmpeg.exe
FFmpeg encontrado: True


## PRÉ-PROCESSAMENTO COMANDOS U,D,L,R

In [ ]:
# 1. CONFIGURAÇÃO

comandos = ["Up", "Down", "Left", "Right"]
fs_destino = 16000
duracao_maxima = 0.999

pastas_saida = {
    comando: pasta_base / f"{comando} Segmentado Manual"
    for comando in comandos
}

for pasta in pastas_saida.values():
    pasta.mkdir(parents=True, exist_ok=True)

ficheiro_registo = pasta_base / "segmentacao_manual_registo.csv"

if ficheiro_registo.exists():
    registos = pd.read_csv(ficheiro_registo).to_dict("records")
else:
    registos = []

estado = {
    "audio": None,
    "fs": fs_destino,
    "duracao": 0,
    "comando": None,
    "numero": None
}


# 2. CARREGAR GRAVAÇÃO

def carregar_gravacao(comando, numero):

    nome = (
        "Gravação.m4a"
        if numero == 1
        else f"Gravação ({numero}).m4a"
    )

    caminho = pasta_base / comando / nome

    if not caminho.exists():
        print(f"Ficheiro não encontrado: {caminho}")
        return

    caminho_temp = pasta_base / "_temp_manual.wav"

    resultado = subprocess.run(
        [
            str(ffmpeg_path),
            "-y",
            "-i", str(caminho),
            "-ac", "1",
            "-ar", str(fs_destino),
            str(caminho_temp)
        ],
        capture_output=True,
        text=True
    )

    if resultado.returncode != 0:
        print(resultado.stderr)
        return

    audio, fs = sf.read(caminho_temp)
    caminho_temp.unlink(missing_ok=True)

    estado.update({
        "audio": audio,
        "fs": fs,
        "duracao": len(audio) / fs,
        "comando": comando,
        "numero": numero
    })

    atualizar_interface()


# 3. REGISTAR RESULTADOS

def atualizar_registo():

    pd.DataFrame(registos).to_csv(
        ficheiro_registo,
        index=False,
        encoding="utf-8-sig"
    )



# 4. ATUALIZAR GRÁFICO

def atualizar_interface():

    with output:
        clear_output(wait=True)

        audio = estado["audio"]
        fs = estado["fs"]
        duracao = estado["duracao"]

        print(
            f"Comando: {estado['comando']} | "
            f"Gravação: {estado['numero']}"
        )

        print(f"Duração original: {duracao:.3f} s")

        intervalo.max = duracao

        fim_inicial = min(duracao, duracao_maxima)
        intervalo.value = (0, fim_inicial)

        desenhar_grafico()


def desenhar_grafico():

    audio = estado["audio"]

    if audio is None:
        return

    fs = estado["fs"]
    duracao = estado["duracao"]

    inicio, fim = intervalo.value

    tempo = np.arange(len(audio)) / fs

    with output:
        clear_output(wait=True)

        print(
            f"Comando: {estado['comando']} | "
            f"Gravação: {estado['numero']}"
        )

        fig, ax = plt.subplots(figsize=(12, 3))

        ax.plot(tempo, audio, linewidth=0.7)

        ax.axvspan(
            inicio,
            fim,
            color="orange",
            alpha=0.3
        )

        ax.set_xlabel("Tempo (s)")
        ax.set_ylabel("Amplitude")
        ax.set_title("Forma de onda da gravação")
        ax.set_xlim(0, duracao)
        ax.grid(True)

        plt.show()
        plt.close(fig)

        print(
            f"Intervalo: {inicio:.3f} – {fim:.3f} s | "
            f"Duração: {fim-inicio:.3f} s"
        )


def atualizar_intervalo(alteracao):

    desenhar_grafico()



# 5. OUVIR SEGMENTO

def ouvir_segmento(_):

    if estado["audio"] is None:
        return

    inicio, fim = intervalo.value

    fs = estado["fs"]
    audio = estado["audio"]

    inicio_amostra = round(inicio * fs)
    fim_amostra = round(fim * fs)

    segmento = audio[inicio_amostra:fim_amostra]

    with output_audio:
        clear_output(wait=True)

        if len(segmento) == 0:
            print("Intervalo vazio.")
            return

        print(
            f"Segmento selecionado: "
            f"{inicio:.3f} – {fim:.3f} s "
            f"({fim-inicio:.3f} s)"
        )

        display(Audio(segmento, rate=fs))


# 6. GUARDAR SEGMENTO

def guardar_segmento(_):

    if estado["audio"] is None:
        return

    inicio, fim = intervalo.value
    duracao = fim - inicio

    if duracao <= 0:
        print("Seleciona um intervalo válido.")
        return

    if duracao >= 1:
        print("O segmento tem de ter menos de 1 segundo.")
        return

    fs = estado["fs"]

    segmento = estado["audio"][
        round(inicio * fs):round(fim * fs)
    ]

    comando = estado["comando"]
    numero = estado["numero"]

    nome_saida = f"Gravacao_{comando}_{numero}.wav"
    caminho_saida = pastas_saida[comando] / nome_saida

    if caminho_saida.exists():
        print("Este segmento já existe. Não foi substituído.")
        return

    sf.write(caminho_saida, segmento, fs)

    registos.append({
        "Comando": comando,
        "Gravacao": numero,
        "Estado": "Aceite",
        "Inicio_s": inicio,
        "Fim_s": fim,
        "Duracao_s": duracao,
        "Ficheiro": str(caminho_saida)
    })

    atualizar_registo()

    print(f"Segmento guardado: {nome_saida}")
    print(f"Duração: {duracao:.3f} s")



# 7. REJEITAR GRAVAÇÃO

def rejeitar_gravacao(_):

    if estado["audio"] is None:
        return

    comando = estado["comando"]
    numero = estado["numero"]

    registos.append({
        "Comando": comando,
        "Gravacao": numero,
        "Estado": "Rejeitada",
        "Inicio_s": np.nan,
        "Fim_s": np.nan,
        "Duracao_s": np.nan,
        "Ficheiro": ""
    })

    atualizar_registo()

    print(f"Gravação {comando} {numero} rejeitada.")



# 8. SELEÇÃO DAS GRAVAÇÕES

opcoes = []

for comando in comandos:

    pasta = pasta_base / comando

    for numero in range(1, 11):

        nome = (
            "Gravação.m4a"
            if numero == 1
            else f"Gravação ({numero}).m4a"
        )

        if (pasta / nome).exists():

            opcoes.append((
                f"{comando} - Gravação {numero}",
                (comando, numero)
            ))


def mudar_gravacao(alteracao):

    comando, numero = alteracao["new"]
    carregar_gravacao(comando, numero)



# 9. INTERFACE

seletor = widgets.Dropdown(
    options=opcoes,
    description="Gravação:",
    layout=widgets.Layout(width="350px")
)

intervalo = widgets.FloatRangeSlider(
    value=(0, 0.5),
    min=0,
    max=1,
    step=0.005,
    description="Intervalo:",
    continuous_update=False,
    readout_format=".3f",
    layout=widgets.Layout(width="700px")
)

botao_ouvir = widgets.Button(
    description="Ouvir segmento",
    button_style="info",
    icon="play"
)

botao_guardar = widgets.Button(
    description="Guardar segmento",
    button_style="success",
    icon="save"
)

botao_rejeitar = widgets.Button(
    description="Rejeitar gravação",
    button_style="danger",
    icon="times"
)

output = widgets.Output()
output_audio = widgets.Output()

seletor.observe(mudar_gravacao, names="value")
intervalo.observe(atualizar_intervalo, names="value")

botao_ouvir.on_click(ouvir_segmento)
botao_guardar.on_click(guardar_segmento)
botao_rejeitar.on_click(rejeitar_gravacao)

display(seletor)
display(intervalo)

display(widgets.HBox([
    botao_ouvir,
    botao_guardar,
    botao_rejeitar
]))

display(output_audio)
display(output)

# Carregar primeira gravação
if opcoes:
    carregar_gravacao(*opcoes[0][1])
else:
    print("Não foram encontradas gravações.")
